Step 1 : Configuration

In [1]:
!pip install groq -q
print("Libraries installed successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.9 MB/s eta 0:00:00
Libraries installed successfully


Step 2 : import all required libraries


In [2]:
import pandas as pd
import sqlite3
import os
from groq import Groq
import re
print("imported successfully")

imported successfully


In [3]:
os.environ["GROQ_API_KEY"] = "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"

client = Groq(api_key=os.environ["GROQ_API_KEY"])

MODEL = "llama-3.1-8b-instant"

print("Groq client initialized successfully")
print(f"Using model: {MODEL}")

Groq client initialized successfully
Using model: llama-3.1-8b-instant


In [4]:
import io

csv_data = """student_id,name,age,gender,subject,marks,attendance,grade
1,Aarav Sharma,20,Male,Mathematics,88,92,A
2,Priya Patel,21,Female,Science,76,85,B
3,Rohan Mehta,20,Male,Programming,95,98,A+
4,Sneha Iyer,22,Female,Mathematics,62,78,C
5,Arjun Nair,21,Male,Programming,91,94,A+
6,Divya Krishnan,20,Female,Science,83,88,A
7,Karan Singh,22,Male,Mathematics,74,81,B
8,Ananya Gupta,21,Female,Programming,89,96,A
9,Vikram Reddy,20,Male,Science,70,79,B
10,Pooja Sharma,22,Female,Mathematics,55,72,D
11,Aditya Kumar,21,Male,Programming,97,99,A+
12,Meera Nambiar,20,Female,Science,81,87,A
13,Rahul Desai,22,Male,Mathematics,68,80,C
14,Kavitha Rajan,21,Female,Programming,86,93,A
15,Nikhil Verma,20,Male,Science,77,84,B
16,Swathi Pillai,22,Female,Mathematics,90,95,A+
17,Manish Joshi,21,Male,Programming,73,82,B
18,Lavanya Menon,20,Female,Science,66,76,C
19,Suresh Babu,22,Male,Mathematics,82,89,A
20,Anjali Singh,21,Female,Programming,94,97,A+
21,Deepak Nair,20,Male,Science,79,86,B
22,Rekha Sharma,22,Female,Mathematics,58,73,D
23,Sanjay Patel,21,Male,Programming,88,91,A
24,Usha Iyer,20,Female,Science,84,90,A
25,Vijay Kumar,22,Male,Mathematics,71,83,B
26,Nandita Rao,21,Female,Programming,92,96,A+
27,Ashok Reddy,20,Male,Science,65,77,C
28,Sunita Gupta,22,Female,Mathematics,87,93,A
29,Ravi Krishnan,21,Male,Programming,78,88,B
30,Bhavna Mehta,20,Female,Science,93,98,A+"""

df = pd.read_csv(io.StringIO(csv_data))

print(f"Dataset loaded : {len(df)} rows, {len(df.columns)} columns")
print("\nFirst 5 rows:")
df.head()

Dataset loaded : 30 rows, 8 columns

First 5 rows:


,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,2,Priya Patel,21,Female,Science,76,85,B
2,3,Rohan Mehta,20,Male,Programming,95,98,A+
3,4,Sneha Iyer,22,Female,Mathematics,62,78,C
4,5,Arjun Nair,21,Male,Programming,91,94,A+


In [5]:
conn = sqlite3.connect("college.db")
df.to_sql("students",conn,if_exists="replace",index=False)

print("Database created: college.db")
print("Table 'students'created with 30 student records")
test_df = pd.read_sql_query("SELECT COUNT(*)as total_rows FROM students",conn)
print(f"\n Verfication:{test_df['total_rows'][0]}rows in database")

Database created: college.db
Table 'students'created with 30 student records

 Verfication:30rows in database


In [9]:
def get_schema(conn,table_name="students"):
  cursor = conn.cursor()
  cursor.execute(f"PRAGMA table_info({table_name})")
  columns= cursor.fetchall()

  schema_lines = [f"Table :{table_name}"]
  schema_lines.append("columns:")

  for col in columns:
    schema_lines.append(f" - {col[1]}({col[2]})")

  cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")
  sample_rows = cursor.fetchall()
  schema_lines.append("]nSample Rows(first 3):")

  for row in sample_rows:
    schema_lines.append(f" {row}")

  return "\n".join(schema_lines)

schema = get_schema(conn)
print(schema)

Table :students
columns:
 - student_id(INTEGER)
 - name(TEXT)
 - age(INTEGER)
 - gender(TEXT)
 - subject(TEXT)
 - marks(INTEGER)
 - attendance(INTEGER)
 - grade(TEXT)
]nSample Rows(first 3):
 (1, 'Aarav Sharma', 20, 'Male', 'Mathematics', 88, 92, 'A')
 (2, 'Priya Patel', 21, 'Female', 'Science', 76, 85, 'B')
 (3, 'Rohan Mehta', 20, 'Male', 'Programming', 95, 98, 'A+')


In [17]:
def generate_sql(user_question,schema_text,client,model):
  system_prompt = f"""You are an expert SQL assistant.
  You are connected to a SQLite database with the following structure:
    {schema_text}
      Rules you must follow:
    1. Generate ONLY a valid SQLite SQL query.
    2. Do not include any explanation or text — only the SQL query.
    3. Do not use markdown code blocks. Return the raw SQL only.
    4. The table name is: students
    5. Only use column names that exist in the schema above.
    6. Use single quotes for string values in WHERE clauses (example: WHERE subject = 'Programming').
    7. If the user asks for top N, use ORDER BY marks DESC LIMIT N.
    """
  response=client.chat.completions.create(
      model=model,
      messages=[
          {"role":"system","content":system_prompt},
          {"role":"user","content":user_question}
      ],
      temperature=0
  )
  sql_query=response.choices[0].message.content.strip()
  return sql_query
question="Show me all female students"
print(f"{question}")
sql=generate_sql(question,schema,client,MODEL)
print(sql)

Show me all female students
SELECT * FROM students WHERE gender = 'Female'


In [21]:
def execute_sql(sql_query, conn):
  clean_sql = sql_query.strip()

  clean_sql = re.sub(r'```sql\s','',clean_sql)
  clean_sql = re.sub(r'```','',clean_sql)
  clean_sql = clean_sql.strip()

  try:
    result_df = pd.read_sql_query(clean_sql, conn)
    return result_df, None

  except Exception as e:
    return None, str(e)
print(f"Executing  SQL :{sql}")
result,error = execute_sql(sql,conn)
if error:
  print(f"Error:{error}")
else:
  print(f"\nQuery returned:{len(result)} rows")
  print(result)

Executing  SQL :SELECT * FROM students WHERE gender = 'Female'

Query returned:15 rows
    student_id            name  age  gender      subject  marks  attendance  \
0            2     Priya Patel   21  Female      Science     76          85   
1            4      Sneha Iyer   22  Female  Mathematics     62          78   
2            6  Divya Krishnan   20  Female      Science     83          88   
3            8    Ananya Gupta   21  Female  Programming     89          96   
4           10    Pooja Sharma   22  Female  Mathematics     55          72   
5           12   Meera Nambiar   20  Female      Science     81          87   
6           14   Kavitha Rajan   21  Female  Programming     86          93   
7           16   Swathi Pillai   22  Female  Mathematics     90          95   
8           18   Lavanya Menon   20  Female      Science     66          76   
9           20    Anjali Singh   21  Female  Programming     94          97   
10          22    Rekha Sharma   22  Female 

In [24]:
def text_to_sql_agent(user_question,conn,client,model,verbose=True):
  print("="*60)
  print(f"USER QUESRION:{user_question}")
  print("="*60)

  if verbose:
    print("\n[STEP 1]reading database schema...")

  schema_text = get_schema(conn)

  if verbose:
    print("schema loaded sucessfully")

  if verbose:
    print("\n[STEP 2]Generating SQL query with Groq LLM...")

  generated_sql = generate_sql(user_question,schema_text,client,model)
  if verbose:
    print(f"Generated SQL :\n {generated_sql}")

  if verbose:
    print("\n[STEP 3]Executing SQL on the database...")

  result_df,error = execute_sql(generated_sql,conn)
  if error:
    print(f"SQL Execution Error:{error}")
    return None,generated_sql
  if verbose:
    print(f"\n[STEP 4]Query returned{len(result_df)} rows(s)")
    print("\nRESULTS:")
    print("-"*40)
    print(result_df.to_string(index=False))

  print("="*60)

  return result_df, generated_sql
result,sql_used = text_to_sql_agent("show top 5 students in programming",conn,client,MODEL)


USER QUESRION:show top 5 students in programming

[STEP 1]reading database schema...
schema loaded sucessfully

[STEP 2]Generating SQL query with Groq LLM...
Generated SQL :
 SELECT name FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5

[STEP 3]Executing SQL on the database...

[STEP 4]Query returned5 rows(s)

RESULTS:
----------------------------------------
        name
Aditya Kumar
 Rohan Mehta
Anjali Singh
 Nandita Rao
  Arjun Nair


In [25]:
result1,_ = text_to_sql_agent("show me all the students who study mathematics",conn,client,MODEL)

USER QUESRION:show me all the students who study mathematics

[STEP 1]reading database schema...
schema loaded sucessfully

[STEP 2]Generating SQL query with Groq LLM...
Generated SQL :
 SELECT * FROM students WHERE subject = 'Mathematics'

[STEP 3]Executing SQL on the database...

[STEP 4]Query returned10 rows(s)

RESULTS:
----------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
          1  Aarav Sharma   20   Male Mathematics     88          92     A
          4    Sneha Iyer   22 Female Mathematics     62          78     C
          7   Karan Singh   22   Male Mathematics     74          81     B
         10  Pooja Sharma   22 Female Mathematics     55          72     D
         13   Rahul Desai   22   Male Mathematics     68          80     C
         16 Swathi Pillai   22 Female Mathematics     90          95    A+
         19   Suresh Babu   22   Male Mathematics     82          89     A
         22  Rekha Sharma   22 Fe

In [26]:
result2,_ = text_to_sql_agent("20 age la iruka students la yaru nu sollu",conn,client,MODEL)

USER QUESRION:20 age la iruka students la yaru nu sollu

[STEP 1]reading database schema...
schema loaded sucessfully

[STEP 2]Generating SQL query with Groq LLM...
Generated SQL :
 SELECT * FROM students WHERE age = 20

[STEP 3]Executing SQL on the database...

[STEP 4]Query returned11 rows(s)

RESULTS:
----------------------------------------
 student_id           name  age gender     subject  marks  attendance grade
          1   Aarav Sharma   20   Male Mathematics     88          92     A
          3    Rohan Mehta   20   Male Programming     95          98    A+
          6 Divya Krishnan   20 Female     Science     83          88     A
          9   Vikram Reddy   20   Male     Science     70          79     B
         12  Meera Nambiar   20 Female     Science     81          87     A
         15   Nikhil Verma   20   Male     Science     77          84     B
         18  Lavanya Menon   20 Female     Science     66          76     C
         21    Deepak Nair   20   Male     Sc